# 🗓 Calendar Agent — Kaggle LoRA Training

Colab 대안. Kaggle은 별도 GPU quota를 제공 (주 30h, T4×2 또는 P100 16GB).

## 실행 전 준비 (한 번만)

### 1. Kaggle 설정 (우측 패널)
- **Accelerator**: `GPU T4 x2` 또는 `GPU P100` 선택
- **Internet**: `On` (외부 패키지·HF 모델 다운로드용)
- **Persistence**: `Variables and Files` (선택 — 세션 재시작 시 일부 보존)

### 2. 데이터 업로드 (Kaggle Dataset으로)
1. https://www.kaggle.com/datasets → `New Dataset`
2. 3개 파일 업로드:
   - `train.jsonl` (로컬 `D:\calendar-agent\data\processed\train.jsonl`)
   - `val.jsonl` (동)
   - `golden.jsonl` (로컬 `D:\calendar-agent\data\eval\golden.jsonl`)
3. Dataset 이름: `calendar-messages` (또는 임의)
4. Visibility: **Private**
5. Create

### 3. 이 노트북에 Dataset 첨부
- 우측 패널 `+ Add Input` → `Datasets` 탭 → 위에서 만든 dataset 검색 → 첨부
- 첨부되면 `/kaggle/input/datasets/sooryongbyun/calendar-messages/` 경로에 파일 보임

### 4. GitHub PAT 발급 (Private repo clone용)
https://github.com/settings/tokens/new?type=classic — repo scope, 7 days

---

준비 끝나면 셀을 위에서 아래로 순서대로 실행. 예상 시간: ~1시간.

## 0. GPU 확인

`Tesla T4` 또는 `Tesla P100` 표시되어야 함. 안 보이면 우측 패널 Accelerator 다시.

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private)

Kaggle은 working dir이 `/kaggle/working/`. 그 안에 clone.

In [ ]:
import os, getpass

%cd /kaggle/working
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/sooryong/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, force-syncing to origin/main…')
    !cd calendar-agent && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/calendar-agent
!git log --oneline -1

## 1.5 라운드 설정 (CONFIG)

clone 직후 바로 현재 라운드를 표시한다. **라운드 변경은 이 셀의 `CONFIG` 한 줄만** 바꾸면 되고, 이후 사전점검·학습·평가 셀이 모두 이 `CONFIG`/`_cfg`/`_mcfg`를 재사용한다.


In [ ]:
# ── 라운드 설정 (CONFIG 한 줄로 라운드 결정 — 이후 모든 셀이 재사용) ──
#   0.5B(fallback): configs/train.yaml  /  Qwen3-0.6B: configs/train_qwen3_0_6b.yaml
CONFIG = 'configs/train_qwen3_0_6b.yaml'
import yaml
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
print(f"★ 현재 라운드 : {_cfg['run_name']}")
print(f"  베이스 모델 : {_mcfg['hf_id']}")
print(f"  output_dir : {_cfg['output_dir']}")
print(f"  유효 배치   : {_cfg['per_device_train_batch_size'] * _cfg['gradient_accumulation_steps']} (T4x2 DDP accum 자동분할)")


## 2. 학습 의존성 설치 + Colab/Kaggle 호환 cleanup

Colab과 동일 — install + torchao 제거.

In [ ]:
# 설치. ⚠ Kaggle 사전설치 RAPIDS(cudf/cuml/dask-cuda)가 numba/cuda-core를 옛 버전에 고정해 둬서
#   "X requires ... which is incompatible" 충돌 경고가 뜨지만, 우리 학습은 RAPIDS 미사용 → 무해.
#   아래 grep으로 그 충돌 보고 줄만 숨김(실제 설치 실패 메시지는 패턴이 달라 그대로 보임).
!pip install -q -e .[train] 2>&1 | grep -vE "which is incompatible|pip's dependency resolver does not|source of the following dependency"
!pip uninstall torchao -y -q   # PEFT가 거부하는 구버전 torchao 제거 (LoRA엔 불필요)
import os, torch
os.environ["WANDB_DISABLED"] = "true"
# CUDA_VISIBLE_DEVICES 설정 안 함 — DDP(torchrun)가 2 GPU를 다 써야 함.
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 3. 데이터 확인 (repo에 포함 — clone으로 따라옴, Kaggle 데이터셋 불필요)

In [ ]:
# 데이터는 repo(git)에 포함 → clone으로 따라옴 (현 라운드 train + golden 50). 데이터셋 첨부 불필요.
for p in ["data/processed/train.jsonl", "data/processed/val.jsonl", "data/eval/golden.jsonl"]:
    n = sum(1 for _ in open(p, encoding="utf-8"))
    print(f"{p}: {n} records")

## 5. 학습 실행

라운드는 위 **'1.5 라운드 설정'** 셀의 `CONFIG`로 결정된다. 아래 셀은 학습 직전 데이터 건수 최종 확인 → 이어서 학습 실행.

T4×2 DDP(torchrun)로 ~1시간. Kaggle 세션 한도 9시간 — 무관.


In [ ]:
# ── 학습 직전 최종 확인 (CONFIG/_cfg/_mcfg는 위 '1.5 라운드 설정' 셀에서 정의) ──
_n = sum(1 for l in open(_cfg['train_data'], encoding='utf-8') if l.strip())
print(f"★ 라운드 {_cfg['run_name']} | base {_mcfg['hf_id']} | {_cfg['train_data']}: {_n}건")


## 5.1 사전점검 — 학습/추론 프롬프트 정렬 확인 (학습 전 권장)

Qwen3는 thinking 모델이라 chat-template이 assistant 턴에 빈 `<think></think>`를 넣는다(**정상·의도된 설계**). 핵심은 *제거*가 아니라 **학습 렌더 == 추론 프롬프트 + gold JSON** 정렬이다 — `eval_model.infer`가 `enable_thinking=False`로 같은 빈 think를 프리필하므로 맞아야 한다.
'라운드 확인' 셀을 먼저 실행한 뒤 이 셀을 돌려라. `OK` 면 학습 진행, `assert` 멈추면 분포 불일치(템플릿 점검).


In [ ]:
# ── 사전점검: 학습 렌더 == 추론 프롬프트 + gold JSON (Qwen3 non-thinking 정렬) ──
#   '1.5 라운드 설정' 셀을 먼저 실행해 _mcfg를 채운 뒤 이 셀 실행.
from transformers import AutoTokenizer
import json
_tok = AutoTokenizer.from_pretrained(_mcfg['hf_id'], trust_remote_code=_mcfg.get('trust_remote_code', False))
_user = "<채널: KakaoTalk>\n<메시지>\n내일 3시 회의\n</메시지>"
_gold = json.dumps({'has_schedule': True, 'events': []}, ensure_ascii=False)
_sys  = _mcfg['system_prompt']
_full = [{'role':'system','content':_sys}, {'role':'user','content':_user}, {'role':'assistant','content':_gold}]
_pre  = [{'role':'system','content':_sys}, {'role':'user','content':_user}]
# 학습 렌더 (train_lora.to_chat와 동일)
_train = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
# 추론 렌더 (eval_model.infer와 동일: thinking 모델이면 enable_thinking=False)
_extra = {}
if 'enable_thinking' in (getattr(_tok, 'chat_template', None) or ''):
    _extra['enable_thinking'] = False
_infer = _tok.apply_chat_template(_pre, tokenize=False, add_generation_prompt=True, **_extra)
print('--- 학습 렌더 ---'); print(_train)
print('--- 추론 프롬프트 ---'); print(_infer)
_aligned = _train.startswith(_infer)
_tail = _train[len(_infer):] if _aligned else ''
_json_next = _tail.lstrip().startswith('{')
print()
print('정렬:', _aligned, '| 추론 직후 JSON:', _json_next, '->', 'OK: 학습 진행' if (_aligned and _json_next) else 'FAIL: 분포 불일치')
assert _aligned and _json_next, '학습 렌더가 (추론 프롬프트 + gold JSON)와 불일치 — train/eval 프롬프트 정렬 깨짐'


In [ ]:
# ── 학습 실행 (위 '1.5 라운드 설정' 셀 먼저 실행해 CONFIG 설정) ──
# DDP: T4 2개를 진짜로 병렬 사용 (torchrun, DataParallel 아님)
print(f"학습 시작 → {CONFIG} (라운드: {_cfg['run_name']})")
!torchrun --nproc_per_node=2 scripts/train_lora.py --config {CONFIG}

## 5.5 학습 직후 평가 (Kaggle GPU, 선택)

merge 후 골든셋으로 바로 평가해 `time_match_rate`/`final_score`를 확인한다. 로컬 CPU 평가(약 14분)보다 GPU가 훨씬 빠르다. 결과 json은 아래 6번 zip에 포함된다.

In [ ]:
# CONFIG에서 모든 경로 자동 인식 — base는 model_config의 hf_id에서 읽음.
import yaml, os
_cfg = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE = _mcfg['hf_id']                          # model_config의 hf_id 자동
LORA_DIR = _cfg['output_dir']                  # 예: models/lora/r12-qwen0.5b
NAME = os.path.basename(LORA_DIR)              # r12-qwen0.5b
MERGED_DIR = f'models/merged/{NAME}'
EVAL_JSON = f'logs/eval_{NAME}.json'
GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
ZIP = f'/kaggle/working/lora_{NAME}.zip'       # 모델별로 구분
print(f'base={BASE}\nlora={LORA_DIR}\ngolden={GOLDEN}\nzip={ZIP}')

# merge → 실 골든셋 평가 (Kaggle GPU)
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 5.6 (선택) 학습 가중치(어댑터) HF 백업 — 세션 만료 대비

Kaggle 세션은 시간이 지나면 사라진다. 학습+eval 직후 **어댑터(40MB)와 eval 결과를 HF private repo에 백업**하면, 나중에 6.5에서 불러와 재학습 없이 Section 7(양자화·검증)을 돌릴 수 있다.
- 인증: Kaggle **Secrets `HF_TOKEN`**(Add-ons→Secrets) 우선, 없으면 getpass 입력. **토큰은 코드/채팅에 안 남는다.**


In [ ]:
# ── 5.6 (선택) 어댑터 HF 백업: 어댑터(40MB) + eval json → HF private repo ──
HF_REPO = 'sooryong9885/Calenda-Qwen3-0.6B'   # private, 라운드별 폴더(=NAME)
DO_BACKUP = True   # False면 건너뜀
if DO_BACKUP:
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        from getpass import getpass; _tok = getpass('HF write token: ')
    from huggingface_hub import HfApi, login
    login(token=_tok)
    api = HfApi()
    api.create_repo(HF_REPO, repo_type='model', private=True, exist_ok=True)
    api.upload_folder(folder_path=LORA_DIR, repo_id=HF_REPO, path_in_repo=NAME,
                      repo_type='model', commit_message=f'{NAME} adapter')
    if os.path.isfile(EVAL_JSON):
        api.upload_file(path_or_fileobj=EVAL_JSON, path_in_repo=f'{NAME}/eval.json',
                        repo_id=HF_REPO, repo_type='model')
    print(f'백업 완료 -> https://huggingface.co/{HF_REPO}/tree/main/{NAME}')
else:
    print('DO_BACKUP=False -> 백업 건너뜀')


## 6. 결과 패키징 (Kaggle Output 자동 노출)

Kaggle은 `/kaggle/working/` 안의 파일을 노트북 우측 패널 `Output`에서 다운로드 가능.
zip을 `/kaggle/working/` 루트에 두면 됨.

In [ ]:
!ls -la {LORA_DIR}/
!zip -r {ZIP} {LORA_DIR} {CONFIG} configs/lora.yaml {_cfg['model_config']} {EVAL_JSON}
!ls -la {ZIP}

## 다운로드 방법

**FileLink는 Kaggle 프록시에서 동작하지 않는다(404 남).** 아래 방법을 쓸 것:

1. **우상단 `Save Version` → `Quick Save`** (재실행 안 함, 현재 `/kaggle/working` 스냅샷) → 저장되면 버전 열기 → **Output** 탭에서 `lora_<라운드>.zip` 다운로드  ← 가장 확실
2. 또는 에디터 우측 **Output(`/kaggle/working`) 파일 패널**에서 `lora_<라운드>.zip` 다운로드 (안 보이면 새로고침)
3. 또는 로컬에서 Kaggle API: `kaggle kernels output <user>/<notebook-slug> -p .`

학습 직후 평가(아래 5.5)를 돌렸다면 `logs/eval_<라운드>-qwen.json`의 `time_match_rate` / `final_score`를 셀 출력에서 바로 확인할 수 있다.

In [ ]:
# 참고: FileLink는 Kaggle에서 종종 404. 위 '다운로드 방법'의 Quick Save -> Output 사용 권장.
from IPython.display import FileLink
FileLink(ZIP)

## 6.5 (선택) HF에서 어댑터 불러오기 — 재학습 없이 merge → 양자화

학습 세션이 만료돼도, 5.6에서 HF에 백업한 **어댑터로 Section 7을 바로 실행**한다. 새 세션: Section 1(clone)·2(설치) 실행 → 이 셀의 `HF_SRC` 채우기 → Section 7. (Section 3·5·5.5 불필요)
- 어댑터(40MB) 다운로드 → **merge**(베이스+어댑터, ~2~3분) → Section 7-A/7-B
- 비우면(`HF_SRC=''`) 평소 학습→merge 경로 그대로.


In [ ]:
# ── (선택) HF에서 어댑터 불러오기 → merge → 양자화 진입 ──
import os, shutil, yaml
HF_SRC = ''            # 예: 'sooryong9885/Calenda-Qwen3-0.6B'  (비우면 이 셀 건너뜀)
RELOAD_ROUND = ''      # 예: 'r30-qwen3-0.6b'  (비우면 train_qwen3_0_6b.yaml의 run_name 기준)
if HF_SRC:
    _cfg  = yaml.safe_load(open('configs/train_qwen3_0_6b.yaml', encoding='utf-8'))
    _mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
    NAME = RELOAD_ROUND or os.path.basename(_cfg['output_dir'])
    MERGED_DIR = f'models/merged/{NAME}'
    GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
    EVAL_JSON = f'logs/eval_{NAME}.json'
    BASE = _mcfg['hf_id']
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        from getpass import getpass; _tok = getpass('HF token: ')
    from huggingface_hub import snapshot_download, login
    login(token=_tok)
    _local = snapshot_download(repo_id=HF_SRC, repo_type='model', allow_patterns=f'{NAME}/*')
    ADIR = f'{_local}/{NAME}'
    if os.path.isfile(f'{ADIR}/eval.json'):
        os.makedirs('logs', exist_ok=True); shutil.copy(f'{ADIR}/eval.json', EVAL_JSON)
    if not os.path.isdir(MERGED_DIR):
        !python scripts/merge_lora.py --base {BASE} --lora {ADIR} --out {MERGED_DIR}
    print('HF 어댑터 -> merge 완료 ->', MERGED_DIR, '| NAME=', NAME, '| FP16:', os.path.isfile(EVAL_JSON))
    print('-> Section 7-A(양자화) -> 7-B(검증)')
else:
    print('HF_SRC 비어있음 -> 평소 학습->merge 경로 진행')


## 7. (선택) 양자화 + GGUF 골든 검증 — 원할 때만 실행

§5.5 eval 셀이 만든 `MERGED_DIR`(merged FP16)을 GGUF로 양자화하고 골든셋으로 검증한다. §1~§6만 돌리면 이 섹션은 실행되지 않는다(배포용 GGUF가 필요할 때만).
- **7-A** 양자화: merged → Q4_K_M + Q8_0 (다운로드는 `/kaggle/working` 루트에 복사)
- **7-B** 검증: 두 양자화를 골든셋으로 채점 → FP16(§5.5) 대비 손실 비교


In [ ]:
# ── 7-A. (선택) 양자화: merged → GGUF Q4_K_M + Q8_0 ──
import os
LLAMA = '/kaggle/working/llama.cpp'
GGUF_DIR = f'models/gguf/{NAME}'; os.makedirs(GGUF_DIR, exist_ok=True)
# llama.cpp 소스(convert용) + llama-quantize 빌드(양자화 바이너리용)
if not os.path.isdir(LLAMA):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git {LLAMA}
!pip install -q gguf
if not os.path.isfile(f'{LLAMA}/build/bin/llama-quantize'):
    !cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /tmp/llama_cfg.log 2>&1
    !cmake --build {LLAMA}/build -j --target llama-quantize > /tmp/llama_build.log 2>&1
QUANT = f'{LLAMA}/build/bin/llama-quantize'
print('quantize 바이너리:', os.path.isfile(QUANT))
F16 = f'{GGUF_DIR}/{NAME}.f16.gguf'
!python {LLAMA}/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16} --outtype f16
for q in ['Q4_K_M', 'Q8_0']:
    !{QUANT} {F16} {GGUF_DIR}/{NAME}.{q}.gguf {q}
!cp {GGUF_DIR}/{NAME}.Q4_K_M.gguf {GGUF_DIR}/{NAME}.Q8_0.gguf /kaggle/working/
!ls -la {GGUF_DIR}


In [ ]:
# ── 7-B. (선택) GGUF 골든 검증: Q4_K_M vs Q8_0 vs FP16(있으면) ──
!pip install -q llama-cpp-python
import json, os
GGUF_DIR = f'models/gguf/{NAME}'
res = {}
for q in ['Q8_0', 'Q4_K_M']:
    outj = f'logs/eval_{NAME}.{q}.json'
    !python scripts/eval_gguf.py --gguf {GGUF_DIR}/{NAME}.{q}.gguf --eval {GOLDEN} --out {outj} --failures_out data/failures/{NAME}.{q}.fail.jsonl --n_gpu_layers 0
    res[q] = json.load(open(outj, encoding='utf-8'))
fp16 = json.load(open(EVAL_JSON, encoding='utf-8')) if os.path.isfile(EVAL_JSON) else None
KEYS = ['final_score','json_valid_rate','title_f1_avg','time_match_rate','location_f1_avg','has_schedule_acc','event_count_acc']
print()
if fp16:
    print(f"{'metric':<20}{'FP16':>9}{'Q8_0':>9}{'Q4_K_M':>9}")
    for k in KEYS:
        print(f"{k:<20}{fp16[k]:>9.3f}{res['Q8_0'][k]:>9.3f}{res['Q4_K_M'][k]:>9.3f}")
else:
    print('(FP16 기준 없음 → Q8_0 vs Q4_K_M만)')
    print(f"{'metric':<20}{'Q8_0':>9}{'Q4_K_M':>9}")
    for k in KEYS:
        print(f"{k:<20}{res['Q8_0'][k]:>9.3f}{res['Q4_K_M'][k]:>9.3f}")
